# Train and Evaluate the Model

- Load the previously created dataset from huggingface

- set up the training arguments

- train

- evaluate

In [2]:
import random
import numpy as np
import torch

seed = 18
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Select device:
if torch.cuda.is_available():
    device = "cuda"
    torch.cuda.manual_seed_all(seed)
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(f"Using device: {device}")


Using device: cpu


# Load the Dataset from HuggingFace

In [3]:
from datasets import load_dataset

dataset = load_dataset("Rogarcia18/symptoms_ner_v00")
# NOTE that the actual labels that will be used for training are under the column: "token_label_ids"
dataset = dataset.rename_column("token_label_ids", "labels")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'labels'],
        num_rows: 14288
    })
    validation: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'labels'],
        num_rows: 1786
    })
    test: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'labels'],
        num_rows: 1786
    })
})

# Setup Training Arguments

In [4]:
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
WANDB_API_KEY = userdata.get('WANDB_API_KEY')

In [5]:
import os
import json
from dotenv import load_dotenv
from transformers import (
    TrainingArguments,
    Trainer,
    DistilBertTokenizerFast,
    DistilBertForTokenClassification,
    DataCollatorForTokenClassification,
)

load_dotenv()
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ.setdefault("WANDB_PROJECT", "symptom-ner")

# Load id2label and label2id mappings from json files
with open("id2label.json", "r") as f:
    id2label = json.load(f)
with open("label2id.json", "r") as f:
    label2id = json.load(f)

# Set up the tokenizer and model
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)
num_labels = len(id2label)
model = DistilBertForTokenClassification.from_pretrained(
    pretrained_model_name_or_path=MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
    ).to(device)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:

OUTPUT_DIR = "distilBERT-symptom-ner-v00"
EVAL_STRATEGY = "epoch"  # simple
# Learning rates to experiment with:
LEARNING_RATES = [5e-5, 3e-5, 1e-5]
# Bs to experiment with:
BS = [10, 16, 32, 64]
NUM_EPOCHS = [20]
run_index = 0

# FIRST ARGUMENTS
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy=EVAL_STRATEGY,
    push_to_hub=True,
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,  # 1 epoch likely underfits; 3 is still quick
    weight_decay=0.01,  # same as in video: https://www.youtube.com/watch?v=ujubwa_oa-0
    warmup_ratio=0.1,  # smooth LR start for small data; drop if you want pure simplicity
    save_strategy="epoch",  # align checkpointing with eval
    logging_strategy="epoch",  # keep logging light
    load_best_model_at_end=True,  # restores best checkpoint
    metric_for_best_model="macro_f1",  # macro_f1 is returned by compute_metrics
    report_to=["wandb"],  # enable Weights & Biases logging
    run_name=f"distilbert-symptom-ner-v00-run-{run_index}",  # shows in W&B runs; adjust as needed
)


# Train !

In [8]:
!pip install seqeval --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [9]:
from metrics import compute_metrics

data_collator = DataCollatorForTokenClassification(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    # call compute_metrics with the argument id2label=id2label
    compute_metrics=lambda eval_pred: compute_metrics(eval_pred, id2label=id2label),
)



/tmp/ipython-input-84879541.py:5: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [10]:
# Uncomment to train
trainer.train()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: robdallagogar (robdallagogar-ferraz-h) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Serializing object of type dict that is 103856 bytes
wandb: WARNING Serializing object of type dict that is 103856 bytes
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Macro F1
1,3.880700,2.792935,[0. 0. 0.33333333 ... 0. 0. 0.5 ],[0. 0. 1. ... 0. 0. 1.],[0. 0. 0.5 ... 0. 0. 0.66666667],0.090652


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TypeError: Object of type ndarray is not JSON serializable

# Evaluate

In [11]:
# Evaluate the model (after training)
metrics = trainer.evaluate()
print(json.dumps(metrics, indent=2))
# Optionally, plot per-label F1 if available
try:
    from metrics import plot_metrics
    plot_path = plot_metrics(metrics, save_path="per_label_f1.png", top_k=30)
    print(f"Saved per-label F1 plot to {plot_path}")
except Exception as exc:
    print(f"Plotting skipped: {exc}")


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Macro F1
1,3.880700,2.792935,[0. 0. 0.33333333 ... 0. 0. 0.5 ],[0. 0. 1. ... 0. 0. 1.],[0. 0. 0.5 ... 0. 0. 0.66666667],0.090652


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TypeError: Object of type ndarray is not JSON serializable